In [0]:
import dlt
import pyspark.sql.functions as F

In [0]:
# Expectations 
rules = {
    "valid_id": "id IS NOT NULL",
    "valid_user_id": "user_id IS NOT NULL",
    "valid_created_at": "created_at IS NOT NULL",
}

In [0]:

@dlt.view(name="FactPostLike_stage")
@dlt.expect_all_or_drop(rules)
def FactPostLike_stage():
    df = (
        spark.readStream.table("travel_journal_catalog.silver.post_likes")
        .withColumn(
            "post_likes_date_id",
            F.date_format(F.col("created_at"), "yyyyMMdd").cast("int"),
        )
    )
    return df

## Fact Table

### Step 2: Fact Table - FactPostLike_stage

In [0]:
@dlt.table(
    name = "fact_post_like",
    table_properties={"quality": "gold"}
)

def fact_post_like():
  """ Joins staging posts_like stream with dim_account to validate keys and builds Gold Fact"""

  like_df    = dlt.read_stream("FactPostLike_stage")
  accounts_df = dlt.read("dim_user")

  fact_post_like_df = like_df.alias("likes").join(
        accounts_df.alias("accounts"),
        (F.col("likes.user_id") == F.col("accounts.user_id"))          # fixed: user_id
        & (F.col("likes.created_at") >= F.col("accounts.__START_AT"))
        & (
            (F.col("likes.created_at") < F.col("accounts.__END_AT"))    # fixed: __END_AT
            | F.col("accounts.__END_AT").isNull()                       # grouped with the line above
        ),
        how="inner",
    ).select(                                                           # chained directly, no stray ')'
        F.col("likes.id").alias("post_like_id"),
        F.col("accounts.DimUserKey").alias("user_key"),
        F.col("likes.user_id"),
        F.col("likes.post_id").alias("trip_post_id"),
        F.col("likes.post_likes_date_id"),                                   
        F.col("likes.created_at"),
        F.lit(1).alias("like_count")
    )

  return fact_post_like_df


## Fact Post Bookmarks

In [0]:

@dlt.view(name="FactPost_Bookmarks_stage")
@dlt.expect_all_or_drop(rules)
def FactPost_Bookmarks_stage():
    df = (
        spark.readStream.table("travel_journal_catalog.silver.post_bookmarks")
        .withColumn(
            "post_bookmarks_date_id",
            F.date_format(F.col("created_at"), "yyyyMMdd").cast("int"),
        )
    )
    return df

In [0]:
@dlt.table(name="fact_post_bookmarks", table_properties={"quality": "gold"})


def fact_post_bookmark():
  """ Joins staging posts_like stream with dim_account to validate keys and builds Gold Fact"""

  bookmarks_df    = dlt.read_stream("FactPost_Bookmarks_stage")
  accounts_df = dlt.read("dim_user")

  fact_post_bookmarks_df = bookmarks_df.alias("bookmarks").join(
        accounts_df.alias("accounts"),
        (F.col("bookmarks.user_id") == F.col("accounts.user_id"))          # fixed: user_id
        & (F.col("bookmarks.created_at") >= F.col("accounts.__START_AT"))
        & (
            (F.col("bookmarks.created_at") < F.col("accounts.__END_AT"))    # fixed: __END_AT
            | F.col("accounts.__END_AT").isNull()                       # grouped with the line above
        ),
        how="inner",
    ).select(                                                           # chained directly, no stray ')'
        F.col("bookmarks.id").alias("post_bookmark_id"),
        F.col("accounts.DimUserKey").alias("user_key"),
        F.col("bookmarks.user_id"),
        F.col("bookmarks.post_id").alias("trip_post_id"),
        F.col("bookmarks.post_bookmarks_date_id"),                                   
        F.col("bookmarks.created_at"),
        F.lit(1).alias("bookmarks_count")
    )

  return fact_post_bookmarks_df